# NYC Yellow Taxi – PySpark Data Processing Pipeline

**Dataset:** NYC TLC Yellow Taxi trips 2022–2024 (36 Parquet files, 1.81 GB, 119.1M rows)
+ taxi zone lookup (265 rows).

**Pipeline:** load → standardize schema → filter (early) → withColumn → select early
→ broadcast join → groupBy / SQL → write results (Parquet).

## Optimization evidence
| Technique | Evidence |
|---|---|
| Filters early / predicate pushdown | Filters appear in `PushedFilters` of the Parquet scan; Catalyst moved the combined filter below the type casts |
| Column pruning (select early) | `ReadSchema` reads only needed columns; queries needing fewer columns ran faster (2.4 min vs 7 min) |
| Broadcast join | Plan shows `BroadcastHashJoin`, no `Exchange hashpartitioning` → large table not shuffled |
| Single-pass aggregation | Data-quality checks computed in one `agg()` instead of separate `count()`s |
| Shuffle partitions | `spark.sql.shuffle.partitions = 16` (default 200 is too many for a 4-core machine) |
| Output partitioning | Small results written with `coalesce(1)` → 1 file each, no shuffle |

## What did not work (measured, then reverted)
- **Partitioned write of the full clean dataset** to the Windows-mounted folder: 58/144 tasks in 23 min,
  while a full read took 1.5 min → bottleneck was disk I/O through the Windows–Docker bind mount.
- **Caching the full dataset:** 0 B in memory, 5.0 GiB on disk (vs 1.81 GB Parquet) → slower than
  recomputing, so it was unpersisted. Caching small aggregated results worked well.

## Key findings
- Manhattan: 89% of trips, 75% of revenue. Queens: 9% of trips but 23% of revenue (JFK/LaGuardia).
- Airport trips: ~10% of trips but ~28% of revenue every year.
- Peak demand at 18:00; slowest traffic at 15:00 (9.6 mph); fastest at 05:00 (20.5 mph).
- Data quality filters removed 3.29% of trips; rows with missing passenger_count were kept.

In [1]:
import pyspark
print(pyspark.__version__)

4.3.0


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("nyc-taxi-pipeline")                  # name shown in Spark UI
    .master("local[*]")                             # local mode, use all available CPU cores
    .config("spark.driver.memory", "4g")            # memory for driver (= executors in local mode)
    .config("spark.sql.shuffle.partitions", "16")   # default is 200, too many for one machine
    .getOrCreate()                                  # reuse session if one already exists
)
spark

In [3]:
# Absolute paths: kernel's working directory may differ from the notebook's folder
TRIPS_PATH = "/home/jovyan/work/data/raw/yellow"
ZONES_PATH = "/home/jovyan/work/data/raw/taxi_zone_lookup.csv"

trips = spark.read.parquet(TRIPS_PATH)      # reads all 36 files in the folder
zones = spark.read.csv(ZONES_PATH, header=True, inferSchema=True)

In [ ]:
trips.printSchema()
zones.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- se

In [ ]:
trips.count()

119136044

In [ ]:
try:
    trips.write.format("noop").mode("overwrite").save()
except Exception as e:
    # Java stack traces are very long; print only the lines that explain the cause
    for line in str(e).splitlines():
        if "Caused by" in line or "cannot be converted" in line or "Exception:" in line:
            print(line.strip())

: org.apache.spark.SparkException: [FAILED_READ_FILE.PARQUET_COLUMN_DATA_TYPE_MISMATCH] Encountered error while reading file file:///home/jovyan/work/data/raw/yellow/yellow_tripdata_2024-10.parquet. Data type mismatches when reading Parquet column [passenger_count]. Expected Spark type double, actual Parquet type INT64. SQLSTATE: KD001
Caused by: org.apache.spark.sql.execution.datasources.SchemaColumnConvertNotSupportedException: column: [passenger_count], physicalType: INT64, logicalType: double


In [4]:
import os
from functools import reduce
from pyspark.sql import functions as F

# Columns whose types drift across files -> one standard type for all
TYPE_FIXES = {
    "vendorid": "int",
    "passenger_count": "int",
    "ratecodeid": "int",
    "pulocationid": "int",
    "dolocationid": "int",
}

def standardize(df):
    # 1) Normalize column names to lowercase (e.g. Airport_fee -> airport_fee)
    df = df.toDF(*[c.lower() for c in df.columns])
    # 2) Cast drifting columns to the standard type
    for col_name, dtype in TYPE_FIXES.items():
        df = df.withColumn(col_name, F.col(col_name).cast(dtype))
    return df

# Read each file separately, standardize it, then stack them all together
files = sorted(os.listdir(TRIPS_PATH))
dfs = [standardize(spark.read.parquet(f"{TRIPS_PATH}/{f}")) for f in files]
trips = reduce(lambda a, b: a.unionByName(b), dfs)

In [ ]:
trips.printSchema()                                       # 5 columns should now be integer
print(trips.count())                                      # must still be 119136044
trips.write.format("noop").mode("overwrite").save()       # full read - should pass now

root
 |-- vendorid: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

119136044


In [ ]:
trips.show(5, truncate=False)   # first 5 rows; truncate=False shows full values

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecodeid|store_and_fwd_flag|pulocationid|dolocationid|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2022-01-01 00:35:40 |2022-01-01 00:53:29  |2              |3.8          |1         |N                 |142         |236         |1           |14.5       |3.0  |0.5    |3.65     

In [ ]:
(trips
 .select("trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count")
 .summary("min", "25%", "50%", "75%", "max")   # 50% = median
 .show())

+-------+-------------+-------------+--------------+------------+---------------+
|summary|trip_distance|  fare_amount|    tip_amount|total_amount|passenger_count|
+-------+-------------+-------------+--------------+------------+---------------+
|    min|          0.0|-1.33391414E8|        -411.0|     -2567.8|              0|
|    25%|         1.06|          8.5|           0.0|        14.3|              1|
|    50%|          1.8|         12.5|           2.5|       19.32|              1|
|    75%|         3.44|         20.5|           4.0|        28.5|              1|
|    max|    398608.62|    401092.32|1.3339136353E8|   401095.62|              9|
+-------+-------------+-------------+--------------+------------+---------------+



In [ ]:
from pyspark.sql import functions as F

year = F.year("tpep_pickup_datetime")

quality = trips.agg(
    F.count("*").alias("total_rows"),
    # each line: condition -> True/False -> cast to 1/0 -> sum = how many rows match
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("zero_or_negative_distance"),
    F.sum((F.col("fare_amount") <= 0).cast("int")).alias("zero_or_negative_fare"),
    F.sum((F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")).cast("int")).alias("dropoff_before_pickup"),
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("null_passenger_count"),
    F.sum(((year < 2022) | (year > 2024)).cast("int")).alias("date_out_of_range"),
)
quality.show(vertical=True)   # vertical=True prints one metric per line (easier to read)

-RECORD 0------------------------------
 total_rows                | 119136044 
 zero_or_negative_distance | 2123821   
 zero_or_negative_fare     | 1412961   
 dropoff_before_pickup     | 61114     
 null_passenger_count      | 6768891   
 date_out_of_range         | 663       



In [6]:
trips.printSchema()

root
 |-- vendorid: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [5]:
from pyspark.sql import functions as F

# --- Filter thresholds (justified by Step 4 data quality results) ---
START_TS = F.lit("2022-01-01 00:00:00").cast("timestamp_ntz")
END_TS   = F.lit("2025-01-01 00:00:00").cast("timestamp_ntz")   # exclusive upper bound
MAX_DISTANCE_MILES = 100
MAX_FARE_USD = 500

trips_filtered = (
    trips
    # Filter 1: keep only our study period (removes device clock errors, e.g. year 2008 / 2098)
    .filter((F.col("tpep_pickup_datetime") >= START_TS) & (F.col("tpep_pickup_datetime") < END_TS))
    # Filter 2: realistic distance (0 = taxi did not move; 398k miles = data error)
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") <= MAX_DISTANCE_MILES))
    # Filter 3: realistic money values (removes refunds/voids and entry errors like -$133M)
    .filter((F.col("fare_amount") > 0) & (F.col("fare_amount") <= MAX_FARE_USD) & (F.col("tip_amount") >= 0))
    # Filter 4: time logic - dropoff must happen after pickup
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

In [6]:
trips_enriched = (
    trips_filtered
    # Trip duration in minutes (timestampdiff works with timestamp_ntz)
    .withColumn("trip_duration_min",
                F.expr("timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime)") / 60.0)
    # Time parts - used later for groupBy and for partitioning the output
    .withColumn("pickup_year",  F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow",   F.date_format("tpep_pickup_datetime", "E"))   # Mon, Tue, ...
    # Average speed in miles per hour
    .withColumn("avg_speed_mph",
                F.round(F.col("trip_distance") / (F.col("trip_duration_min") / 60), 2))
    # Tip % only for card payments (cash tips are not recorded -> NULL, not 0)
    .withColumn("tip_pct",
                F.when(F.col("payment_type") == 1,
                       F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)))
    # Filter 5: sanity check on derived columns (can only run after they exist)
    .filter((F.col("trip_duration_min") >= 1) & (F.col("trip_duration_min") <= 180)
            & (F.col("avg_speed_mph") <= 80))
)

In [9]:
(trips_enriched
 .select("tpep_pickup_datetime", "trip_distance", "trip_duration_min",
         "pickup_hour", "pickup_dow", "avg_speed_mph", "payment_type", "tip_pct")
 .show(5))

+--------------------+-------------+------------------+-----------+----------+-------------+------------+-------+
|tpep_pickup_datetime|trip_distance| trip_duration_min|pickup_hour|pickup_dow|avg_speed_mph|payment_type|tip_pct|
+--------------------+-------------+------------------+-----------+----------+-------------+------------+-------+
| 2022-01-01 00:35:40|          3.8|17.816666666666666|          0|       Sat|         12.8|           1|  25.17|
| 2022-01-01 00:33:43|          2.1|               8.4|          0|       Sat|         15.0|           1|   50.0|
| 2022-01-01 00:53:21|         0.97| 8.966666666666667|          0|       Sat|         6.49|           1|  23.47|
| 2022-01-01 00:25:21|         1.09|10.033333333333333|          0|       Sat|         6.52|           2|   NULL|
| 2022-01-01 00:36:48|          4.3| 37.53333333333333|          0|       Sat|         6.87|           1|  12.77|
+--------------------+-------------+------------------+-----------+----------+----------

In [10]:
kept = trips_enriched.count()
print(f"Kept: {kept:,} rows ({kept / 119_136_044 * 100:.2f}%)")

Kept: 115,217,330 rows (96.71%)


In [11]:
trips_enriched.explain()

== Physical Plan ==
*(1) Union
:- *(1) Project [vendorid#79, tpep_pickup_datetime#60, tpep_dropoff_datetime#61, passenger_count#80, trip_distance#63, ratecodeid#81, store_and_fwd_flag#65, pulocationid#82, dolocationid#83, payment_type#68L, fare_amount#69, extra#70, mta_tax#71, tip_amount#72, tolls_amount#73, improvement_surcharge#74, total_amount#75, congestion_surcharge#76, airport_fee#77, trip_duration_min#1624, pickup_year#1625, pickup_month#1626, pickup_hour#1627, pickup_dow#1628, round((trip_distance#63 / (trip_duration_min#1624 / 60.0)), 2) AS avg_speed_mph#1629, ... 1 more fields]
:  +- *(1) Project [cast(VendorID#40L as int) AS vendorid#79, tpep_pickup_datetime#41 AS tpep_pickup_datetime#60, tpep_dropoff_datetime#42 AS tpep_dropoff_datetime#61, cast(passenger_count#43 as int) AS passenger_count#80, trip_distance#44 AS trip_distance#63, cast(RatecodeID#45 as int) AS ratecodeid#81, store_and_fwd_flag#46 AS store_and_fwd_flag#65, cast(PULocationID#47L as int) AS pulocationid#82, c

In [12]:
import io, contextlib

# explain() prints to the screen; capture that text instead so we can search it
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    trips_enriched.explain("formatted")   # "formatted" mode lists details of each operator

# Show only the details of the FIRST file scan (all 36 scans are the same)
for line in buffer.getvalue().splitlines():
    if "PushedFilters" in line or "ReadSchema" in line:
        print(line[:600])   # first 600 characters is enough
        if "ReadSchema" in line:
            break           # stop after the first file

PushedFilters: [IsNotNull(tpep_pickup_datetime), IsNotNull(trip_distance), IsNotNull(fare_amount), IsNotNull(tip_amount), IsNotNull(tpep_dropoff_datetime), GreaterThanOrEqual(tpep_pickup_datetime,2022-01-01T00:00), LessThan(tpep_pickup_datetime,2025-01-01T00:00), GreaterThan(trip_distance,0.0), LessThanOrEqual(trip_distance,100.0), GreaterThan(fare_amount,0.0), LessThanOrEqual(fare_amount,500.0), GreaterThanOrEqual(tip_amount,0.0)]
ReadSchema: struct<VendorID:bigint,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:double,trip_distance:double,RatecodeID:double,store_and_fwd_flag:string,PULocationID:bigint,DOLocationID:bigint,payment_type:bigint,fare_amount:double,extra:double,mta_tax:double,tip_amount:double,tolls_amount:double,improvement_surcharge:double,total_amount:double,congestion_surcharge:double,airport_fee:double>


In [7]:
import io, contextlib

def plan_lines(df, keywords, mode="formatted", max_len=300):
    """Print only the lines of the query plan that contain any of the keywords."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    seen = set()
    for line in buffer.getvalue().splitlines():
        line = line.strip()
        # union of 36 files repeats the same lines -> print each unique line once
        if any(k in line for k in keywords) and line not in seen:
            seen.add(line)
            print(line[:max_len])

In [8]:
# Columns we actually need for the analysis (raw + derived)
KEEP_COLS = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "pulocationid", "dolocationid",
    "passenger_count", "trip_distance", "payment_type",
    "fare_amount", "tip_amount", "total_amount",
    "trip_duration_min", "pickup_year", "pickup_month",
    "pickup_hour", "pickup_dow", "avg_speed_mph", "tip_pct",
]

trips_slim = trips_enriched.select(*KEEP_COLS)

In [15]:
plan_lines(trips_slim, ["ReadSchema"], max_len=600)

ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:double,trip_distance:double,PULocationID:bigint,DOLocationID:bigint,payment_type:bigint,fare_amount:double,tip_amount:double,total_amount:double>
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:bigint,trip_distance:double,PULocationID:int,DOLocationID:int,payment_type:bigint,fare_amount:double,tip_amount:double,total_amount:double>


In [9]:
# Pickup version: LocationID -> pulocationid, Borough -> pu_borough, ...
pu_zones = zones.select(
    F.col("LocationID").alias("pulocationid"),
    F.col("Borough").alias("pu_borough"),
    F.col("Zone").alias("pu_zone"),
)

# Dropoff version
do_zones = zones.select(
    F.col("LocationID").alias("dolocationid"),
    F.col("Borough").alias("do_borough"),
    F.col("Zone").alias("do_zone"),
)

In [10]:
trips_joined = (
    trips_slim
    # F.broadcast() = "send a copy of this small table to every worker"
    .join(F.broadcast(pu_zones), on="pulocationid", how="left")
    .join(F.broadcast(do_zones), on="dolocationid", how="left")
)

In [18]:
plan_lines(trips_joined, ["Join", "Exchange"])

+- BroadcastHashJoin LeftOuter BuildRight (156)
:  +- BroadcastHashJoin LeftOuter BuildRight (150)
:     +- BroadcastExchange (149)
+- BroadcastExchange (155)
(149) BroadcastExchange
(150) BroadcastHashJoin
Join type: LeftOuter
Join condition: None
(155) BroadcastExchange
(156) BroadcastHashJoin


In [19]:
check = trips_joined.agg(
    F.count("*").alias("rows_after_join"),
    F.sum(F.col("pu_borough").isNull().cast("int")).alias("missing_pickup_zone"),
    F.sum(F.col("do_borough").isNull().cast("int")).alias("missing_dropoff_zone"),
)
check.show(vertical=True)

-RECORD 0-------------------------
 rows_after_join      | 115217330 
 missing_pickup_zone  | 0         
 missing_dropoff_zone | 0         



In [20]:
trips_joined.select("pulocationid", "pu_borough", "pu_zone",
                    "dolocationid", "do_borough", "do_zone").show(5, truncate=False)

+------------+----------+-----------------------+------------+----------+---------------------+
|pulocationid|pu_borough|pu_zone                |dolocationid|do_borough|do_zone              |
+------------+----------+-----------------------+------------+----------+---------------------+
|142         |Manhattan |Lincoln Square East    |236         |Manhattan |Upper East Side North|
|236         |Manhattan |Upper East Side North  |42          |Manhattan |Central Harlem North |
|166         |Manhattan |Morningside Heights    |166         |Manhattan |Morningside Heights  |
|114         |Manhattan |Greenwich Village South|68          |Manhattan |East Chelsea         |
|68          |Manhattan |East Chelsea           |163         |Manhattan |Midtown North        |
+------------+----------+-----------------------+------------+----------+---------------------+
only showing top 5 rows


In [ ]:
# NOTE: Writing clean data to the Windows-mounted folder was stopped after ~28 min (58/144 tasks).
# A full read (noop) took only ~1.5 min -> bottleneck = disk I/O through the Windows-Docker bind mount.
# Decision: cache the cleaned DataFrame instead; only small aggregated results are written at the end.

#CLEAN_PATH = "/home/jovyan/work/output/trips_clean"

#(trips_joined
 #.write
 #.mode("overwrite")                          # re-running the cell replaces old output (idempotent)
 #.partitionBy("pickup_year", "pickup_month") # one folder per year/month
 #.parquet(CLEAN_PATH))

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/conda/lib/python3.13/socket.py", line 723, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# NOTE: cache() was tested and removed.
# Result: 100% cached, but 0 B in memory and 5.0 GiB on disk (vs 1.81 GB Parquet source).
# Reading the disk cache was slower than recomputing from Parquet -> unpersisted.
trips_joined.cache()

# cache() is lazy - an action is needed to actually fill the cache (~3 min, only once per session)
cached_rows = trips_joined.count()
print(f"Cached rows: {cached_rows:,}")

Cached rows: 115,217,330


In [15]:
borough_summary = (
    trips_joined
    .groupBy("pu_borough")                                       # one group per pickup borough
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_usd"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min"),
        F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct"),       # avg ignores NULLs -> card payments only
        F.round(F.sum("total_amount") / 1e6, 1).alias("revenue_musd"),  # revenue in millions of USD
    )
    .orderBy(F.desc("trips"))                                    # biggest borough first
)

borough_summary.show(truncate=False)

+-------------+---------+------------+---------------+----------------+-----------+------------+
|pu_borough   |trips    |avg_fare_usd|avg_distance_mi|avg_duration_min|avg_tip_pct|revenue_musd|
+-------------+---------+------------+---------------+----------------+-----------+------------+
|Manhattan    |102659139|14.71       |2.5            |14.2            |26.0       |2265.8      |
|Queens       |10375224 |49.7        |12.99          |36.8            |23.9       |700.7       |
|Brooklyn     |1052243  |26.45       |5.62           |27.6            |13.8       |32.9        |
|Unknown      |868543   |21.52       |5.03           |19.2            |24.5       |26.6        |
|Bronx        |202540   |31.0        |7.34           |34.3            |6.8        |7.0         |
|N/A          |53567    |57.11       |10.67          |52.6            |651.4      |3.3         |
|Staten Island|4821     |50.14       |15.32          |45.0            |3.0        |0.3         |
|EWR          |1253     |80.44

In [13]:
# Remove the cached data: it was 100% on disk (5.0 GiB) and slower than recomputing from Parquet
trips_joined.unpersist()

DataFrame[dolocationid: int, pulocationid: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: int, trip_distance: double, payment_type: bigint, fare_amount: double, tip_amount: double, total_amount: double, trip_duration_min: double, pickup_year: int, pickup_month: int, pickup_hour: int, pickup_dow: string, avg_speed_mph: double, tip_pct: double, pu_borough: string, pu_zone: string, do_borough: string, do_zone: string]

In [16]:
plan_lines(borough_summary, ["Exchange", "HashAggregate", "Scan"])

+- Exchange (125)
+- HashAggregate (124)
+- Exchange (123)
+- HashAggregate (122)
:     :  :     +- Scan parquet  (1)
:     :  :     +- Scan parquet  (4)
:     :  :     +- Scan parquet  (7)
:     :  :     +- Scan parquet  (10)
:     :  :     +- Scan parquet  (13)
:     :  :     +- Scan parquet  (16)
:     :  :     +- Scan parquet  (19)
:     :  :     +- Scan parquet  (22)
:     :  :     +- Scan parquet  (25)
:     :  :     +- Scan parquet  (28)
:     :  :     +- Scan parquet  (31)
:     :  :     +- Scan parquet  (34)
:     :  :     +- Scan parquet  (37)
:     :  :     +- Scan parquet  (40)
:     :  :     +- Scan parquet  (43)
:     :  :     +- Scan parquet  (46)
:     :  :     +- Scan parquet  (49)
:     :  :     +- Scan parquet  (52)
:     :  :     +- Scan parquet  (55)
:     :  :     +- Scan parquet  (58)
:     :  :     +- Scan parquet  (61)
:     :  :     +- Scan parquet  (64)
:     :  :     +- Scan parquet  (67)
:     :  :     +- Scan parquet  (70)
:     :  :     +- Scan parquet  (

In [17]:
# Give the DataFrame a name that SQL can use. No data is copied - it is just a name for the plan.
trips_joined.createOrReplaceTempView("trips")

In [18]:
hourly = spark.sql("""
    SELECT
        pickup_hour,
        COUNT(*)                         AS trips,
        ROUND(AVG(avg_speed_mph), 1)     AS avg_speed_mph,     -- lower speed = heavier traffic
        ROUND(AVG(trip_duration_min), 1) AS avg_duration_min,
        ROUND(AVG(fare_amount), 2)       AS avg_fare_usd
    FROM trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""")

hourly.cache()      # result is only 24 rows -> fits in memory easily, reused when writing later
hourly.show(24)     # 24 hours

+-----------+-------+-------------+----------------+------------+
|pickup_hour|  trips|avg_speed_mph|avg_duration_min|avg_fare_usd|
+-----------+-------+-------------+----------------+------------+
|          0|3243986|         14.9|            14.3|       18.63|
|          1|2127826|         14.7|            13.1|       16.89|
|          2|1398087|         14.8|            12.3|       15.84|
|          3| 913518|         15.9|            12.4|       16.84|
|          4| 629196|         19.0|            14.4|       22.26|
|          5| 695531|         20.5|            16.0|       25.56|
|          6|1660664|         16.8|            15.6|       20.97|
|          7|3211485|         13.3|            15.5|       18.04|
|          8|4396319|         11.4|            15.9|       17.34|
|          9|4910443|         11.0|            15.9|       17.14|
|         10|5342850|         10.7|            16.2|       17.26|
|         11|5800205|         10.2|            16.8|       17.49|
|         

In [19]:
airport_by_year = spark.sql("""
    WITH flagged AS (                       -- CTE: step 1 = mark each trip as airport or not
        SELECT
            pickup_year,
            total_amount,
            CASE WHEN pu_zone IN ('JFK Airport', 'LaGuardia Airport', 'Newark Airport')
                   OR do_zone IN ('JFK Airport', 'LaGuardia Airport', 'Newark Airport')
                 THEN 1 ELSE 0 END AS is_airport
        FROM trips
    )
    SELECT                                  -- step 2 = summarize per year
        pickup_year,
        COUNT(*)                                        AS total_trips,
        SUM(is_airport)                                 AS airport_trips,
        ROUND(100.0 * SUM(is_airport) / COUNT(*), 2)    AS airport_trip_pct,
        ROUND(100.0 * SUM(CASE WHEN is_airport = 1 THEN total_amount ELSE 0 END)
              / SUM(total_amount), 2)                   AS airport_revenue_pct
    FROM flagged
    GROUP BY pickup_year
    ORDER BY pickup_year
""")

airport_by_year.cache()
airport_by_year.show()

+-----------+-----------+-------------+----------------+-------------------+
|pickup_year|total_trips|airport_trips|airport_trip_pct|airport_revenue_pct|
+-----------+-----------+-------------+----------------+-------------------+
|       2022|   38627700|      3847426|            9.96|              26.86|
|       2023|   37036072|      4081593|           11.02|              29.61|
|       2024|   39553558|      3982261|           10.07|              27.79|
+-----------+-----------+-------------+----------------+-------------------+



In [20]:
RESULTS_PATH = "/home/jovyan/work/output/results"

# Order matters for speed: the cached results go first (seconds),
# borough_summary is not cached, so it recomputes from source (~7 min)
results = {
    "airport_by_year": airport_by_year,
    "hourly_demand":   hourly,
    "borough_summary": borough_summary,
}

for name, df in results.items():
    (df
     .coalesce(1)               # tiny result -> one file (avoid small files problem, no shuffle)
     .write
     .mode("overwrite")         # safe to re-run
     .parquet(f"{RESULTS_PATH}/{name}"))
    print(f"written: {name}")

written: airport_by_year
written: hourly_demand
written: borough_summary


In [21]:
import os

for name in results:
    path = f"{RESULTS_PATH}/{name}"
    files = [f for f in os.listdir(path) if f.endswith(".parquet")]
    df = spark.read.parquet(path)
    print(f"\n=== {name}: {df.count()} rows, {len(files)} parquet file(s) ===")
    df.show(30, truncate=False)


=== airport_by_year: 3 rows, 1 parquet file(s) ===
+-----------+-----------+-------------+----------------+-------------------+
|pickup_year|total_trips|airport_trips|airport_trip_pct|airport_revenue_pct|
+-----------+-----------+-------------+----------------+-------------------+
|2022       |38627700   |3847426      |9.96            |26.86              |
|2023       |37036072   |4081593      |11.02           |29.61              |
|2024       |39553558   |3982261      |10.07           |27.79              |
+-----------+-----------+-------------+----------------+-------------------+


=== hourly_demand: 24 rows, 1 parquet file(s) ===
+-----------+-------+-------------+----------------+------------+
|pickup_hour|trips  |avg_speed_mph|avg_duration_min|avg_fare_usd|
+-----------+-------+-------------+----------------+------------+
|0          |3243986|14.9         |14.3            |18.63       |
|1          |2127826|14.7         |13.1            |16.89       |
|2          |1398087|14.8  

In [22]:
import time

# Read one month only (smaller data -> quick, clear demo)
jan = spark.read.parquet(f"{TRIPS_PATH}/yellow_tripdata_2024-01.parquet")

In [23]:
start = time.time()

demo = (
    jan
    .filter(F.col("trip_distance") > 0)                                    # transformation
    .filter(F.col("fare_amount") > 0)                                      # transformation
    .withColumn("tip_pct", F.col("tip_amount") / F.col("fare_amount") * 100)  # transformation
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))             # transformation
    .groupBy("pickup_hour")                                                # transformation (wide)
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("tip_pct"), 2).alias("avg_tip_pct"))                # transformation
    .orderBy("pickup_hour")                                                # transformation (wide)
)

print(f"Transformations took: {time.time() - start:.3f} seconds")
print(type(demo))   # a DataFrame = a PLAN, not data yet

Transformations took: 2.379 seconds
<class 'pyspark.sql.classic.dataframe.DataFrame'>


In [24]:
start = time.time()

demo.show(24)   # ACTION -> Spark now reads the file and runs every step above

print(f"Action (show) took: {time.time() - start:.3f} seconds")

+-----------+------+-----------+
|pickup_hour| trips|avg_tip_pct|
+-----------+------+-----------+
|          0| 75241|      20.01|
|          1| 50481|      19.95|
|          2| 34961|      20.24|
|          3| 22942|      36.79|
|          4| 15276|      16.95|
|          5| 17494|      16.65|
|          6| 39415|      17.21|
|          7| 80858|      19.24|
|          8|113484|       32.2|
|          9|125600|       20.0|
|         10|135408|      19.87|
|         11|146732|       19.8|
|         12|159896|      19.74|
|         13|165334|      19.79|
|         14|178011|       19.8|
|         15|183980|       19.8|
|         16|184956|      21.84|
|         17|200289|      22.34|
|         18|206266|      22.78|
|         19|178791|      22.86|
|         20|155544|      22.17|
|         21|155895|      21.88|
|         22|138232|      21.29|
|         23|104628|      20.41|
+-----------+------+-----------+

Action (show) took: 11.201 seconds


In [25]:
start = time.time()

n = demo.count()   # another ACTION -> without cache, Spark recomputes the whole plan again

print(f"Rows: {n}, action (count) took: {time.time() - start:.3f} seconds")

Rows: 24, action (count) took: 4.517 seconds


### Result: transformations are lazy, actions are eager
| Step | Operation | Time | Spark jobs created |
|---|---|---|---|
| 10.2 | 7 transformations (filter, withColumn, groupBy, orderBy) | 2.4 s | **0** |
| 10.3 | `show()` (action) | 11.2 s | 2 |
| 10.4 | `count()` (action) | 4.5 s | 3 |

- Transformations only built and validated the plan on the driver (analysis); no data was read and no job ran.
- Each action triggered jobs and re-read the source file (4/4 tasks, none skipped) – Spark does not keep results between actions.
- The second action was faster because of warm-up (OS file cache, JVM) and because the optimizer removed the unnecessary sort for `count()` – not because of caching.

In [26]:
# Q1 2024 = 3 monthly files (all in the same new format -> no schema drift here)
q1_paths = [f"{TRIPS_PATH}/yellow_tripdata_2024-{m:02d}.parquet" for m in (1, 2, 3)]

q1 = (
    spark.read.parquet(*q1_paths)
    .filter((F.col("trip_distance") > 0) & (F.col("fare_amount") > 0))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("tip_pct", F.col("tip_amount") / F.col("fare_amount") * 100)
    .select("pickup_hour", "trip_distance", "fare_amount", "tip_pct")   # only 4 columns -> small cache
)

def run_query(df):
    """The repeated action: average fare and tip % per hour (returns only 24 rows)."""
    return (df.groupBy("pickup_hour")
              .agg(F.avg("fare_amount"), F.avg("tip_pct"))
              .collect())

def timed(label, fn):
    """Run fn(), print and return how long it took."""
    start = time.time()
    fn()
    elapsed = time.time() - start
    print(f"{label}: {elapsed:.2f} s")
    return elapsed

In [27]:
t1 = timed("No cache   - run 1", lambda: run_query(q1))
t2 = timed("No cache   - run 2", lambda: run_query(q1))

No cache   - run 1: 9.35 s
No cache   - run 2: 2.73 s


In [28]:
q1.cache()   # lazy: only marks q1 to be stored the next time it is computed

t3 = timed("With cache - run 1 (computes + fills cache)", lambda: run_query(q1))
t4 = timed("With cache - run 2 (reads from memory)    ", lambda: run_query(q1))

print(f"\nSpeed-up: {t2 / t4:.1f}x faster (no-cache run 2 vs cached run 2)")

With cache - run 1 (computes + fills cache): 12.45 s
With cache - run 2 (reads from memory)    : 0.84 s

Speed-up: 3.2x faster (no-cache run 2 vs cached run 2)


In [29]:
# The plan should now read from memory (InMemoryTableScan) instead of the Parquet files
plan_lines(q1.groupBy("pickup_hour").count(), ["InMemoryTableScan", "Scan parquet"])

+- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
+- Scan parquet  (3)
(1) InMemoryTableScan
(3) Scan parquet


In [30]:
q1.unpersist()   # free the memory when we no longer need the cached data

DataFrame[pickup_hour: int, trip_distance: double, fare_amount: double, tip_pct: double]

### Result: caching speeds up repeated actions – when the data fits in memory

**Setup:** Q1 2024 (3 monthly files, ~9M rows), filtered and reduced to 4 columns
(`pickup_hour`, `trip_distance`, `fare_amount`, `tip_pct`). The repeated action is a `groupBy("pickup_hour")`
aggregation (average fare and tip %), executed twice without cache and twice with cache.

| Run | Time |
|---|---|
| No cache – run 1 (cold) | 9.35 s |
| No cache – run 2 (warm) | 2.73 s |
| With cache – run 1 (compute + fill cache) | 12.45 s |
| With cache – run 2 (read from memory) | **0.84 s** |

- The cached repeated action was **3.2x faster** than the warm uncached run (**11x** faster than the cold run).
- The first cached run is the slowest: caching is a one-time investment that pays off only when the data is reused.
- The physical plan shows `InMemoryTableScan`, i.e. Spark reads the cached data. The `Scan parquet` below it is the
  cache's **lineage** – it is used only to rebuild lost partitions, not executed while the cache is valid.
- Storage tab: the cached data took **212.4 MiB in memory and 0 B on disk** (100% cached).
  Filters were still pushed down to Parquet and only 4 columns were read (`PushedFilters`, `ReadSchema`).

**Comparison with caching the full dataset**

| | Full dataset | Q1 2024, 4 columns |
|---|---|---|
| Rows × columns | 115M × 21 | ~9M × 4 |
| Size in memory | 0 B | 212.4 MiB |
| Size on disk | 5.0 GiB | 0 B |
| Effect | Slower than recomputing → unpersisted | 3.2x faster repeated actions |

**Lesson:** cache only data that is (1) reused several times and (2) small enough to fit in memory.
Selecting only the needed columns before caching makes the cache much smaller.

In [31]:
# Results are already written to output/results, so the small caches are no longer needed.
# Without the cache, explain() shows the FULL pipeline instead of just "read from memory".
hourly.unpersist()
airport_by_year.unpersist()

DataFrame[pickup_year: int, total_trips: bigint, airport_trips: bigint, airport_trip_pct: decimal(27,2), airport_revenue_pct: double]

In [33]:
import io, contextlib, os

# Capture the full formatted plan as text
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    airport_by_year.explain("formatted")
full_plan = buffer.getvalue()

PLANS_PATH = "/home/jovyan/work/output/plans"
os.makedirs(PLANS_PATH, exist_ok=True)
with open(f"{PLANS_PATH}/airport_by_year_physical_plan.txt", "w") as f:
    f.write(full_plan)
print(f"Full plan saved: {len(full_plan.splitlines())} lines\n")

# Show only the key optimization lines in the notebook
plan_lines(airport_by_year,
           ["BroadcastHashJoin", "BroadcastExchange", "Exchange", "HashAggregate",
            "PushedFilters", "ReadSchema"],
           max_len=250)

Full plan saved: 761 lines

+- Exchange (125)
+- HashAggregate (124)
+- Exchange (123)
+- HashAggregate (122)
+- BroadcastHashJoin LeftOuter BuildRight (120)
:  +- BroadcastHashJoin LeftOuter BuildRight (114)
:     +- BroadcastExchange (113)
+- BroadcastExchange (119)
PushedFilters: [IsNotNull(tpep_pickup_datetime), IsNotNull(trip_distance), IsNotNull(fare_amount), IsNotNull(tip_amount), IsNotNull(tpep_dropoff_datetime), GreaterThanOrEqual(tpep_pickup_datetime,2022-01-01T00:00), LessThan(tpep_pickup_datetime,2025-
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,trip_distance:double,PULocationID:bigint,DOLocationID:bigint,fare_amount:double,tip_amount:double,total_amount:double>
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,trip_distance:double,PULocationID:int,DOLocationID:int,fare_amount:double,tip_amount:double,total_amount:double>
PushedFilters: [IsNotNull(LocationID)]
ReadSchema: struct<LocationID

In [ ]:
# Release cached data and stop the Spark session (frees memory)
spark.stop()